# cycle-detection-temp-set — faded example 3: Complete the path slice that reconstructs the cycle

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `cycle-detection-temp-set`. Running the beacon reports progress on the `Backprop: cycle detection via temp set` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: cycle detection via temp set` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cycle-detection-temp-set`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cycle-detection-temp-set"
DD_SUBTOPIC = "Backprop: cycle detection via temp set"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

To report the actual cycle (not just a boolean), keep an ordered `path` of the vertices on the stack alongside the `on_stack` set. When you hit a back-edge — a child already in `on_stack` — the cycle is the suffix of `path` starting at that child's first appearance, with the child repeated to close the loop. The slice index is found by locating the back-edge target inside `path`.

## Faded exercise 3

Implement `find_cycle(adj, start)` returning the list of nodes around the first cycle (back-edge target repeated at the end), or `None`. The DFS, the `on_stack`/`path` bookkeeping, and the un-stack cleanup are written. **Complete the back-edge handling line** that builds the closed cycle path from `path`.

**Fill in:** Builds the cycle: the suffix of path from the back-edge target's first index onward, with the target node appended again to close the loop.

In [ ]:
def find_cycle(adj, start):
    path, on_stack, perm = [], set(), set()

    def visit(u):
        if u in perm:
            return None
        if u in on_stack:
            i = path.index(u)
            return None  # TODO: return path[i:] plus u appended to close the cycle
        on_stack.add(u)
        path.append(u)
        for v in adj.get(u, []):
            cyc = visit(v)
            if cyc is not None:
                return cyc
        on_stack.discard(u)
        path.pop()
        perm.add(u)
        return None

    return visit(start)


def _test():
    dag = {"a": ["b", "c"], "b": ["d"], "c": ["d"], "d": []}
    assert find_cycle(dag, "a") is None, "diamond DAG has no cycle"
    cyc = {"a": ["b"], "b": ["c"], "c": ["a"]}
    res = find_cycle(cyc, "a")
    assert res == ["a", "b", "c", "a"], f"expected closed cycle, got {res}"
    res2 = find_cycle({"a": ["b"], "b": ["b"]}, "a")
    assert res2 == ["b", "b"], f"self-loop cycle, got {res2}"
    res3 = find_cycle({"s": ["x"], "x": ["y"], "y": ["x"]}, "s")
    assert res3 == ["x", "y", "x"], f"cycle not touching start, got {res3}"


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def find_cycle(adj, start):
    path, on_stack, perm = [], set(), set()

    def visit(u):
        if u in perm:
            return None
        if u in on_stack:
            i = path.index(u)
            return path[i:] + [u]
        on_stack.add(u)
        path.append(u)
        for v in adj.get(u, []):
            cyc = visit(v)
            if cyc is not None:
                return cyc
        on_stack.discard(u)
        path.pop()
        perm.add(u)
        return None

    return visit(start)
```
</details>